# 08 — PCA and Unsupervised Clustering

PCA and K-Means are fit using training cycles only and without diagnosis labels. Labels are joined only afterwards for visualization and ARI/cluster-composition analysis.

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.post_05.common import FEATURES_DIR, FIGURES_DIR, ensure_output_dirs, save_csv_new, new_artifact_path
from src.post_05.mining import pca_and_cluster
ensure_output_dirs()
data=pd.read_csv(FEATURES_DIR/'cycle_features.csv').merge(pd.read_csv(FEATURES_DIR/'patient_splits.csv')[['patient_id','split']],on='patient_id',validate='many_to_one')
selected=pd.read_csv(FEATURES_DIR/'selected_feature_list.csv').feature.tolist()
train=data[data.split.eq('train')].reset_index(drop=True)
result=pca_and_cluster(train,selected)
print('Selected k:',result['best_k']); display(result['scores']); display(result['projection'].groupby(['cluster','diagnosis']).size().unstack(fill_value=0))
save_csv_new(result['scores'],FEATURES_DIR/'cluster_model_selection.csv')
save_csv_new(result['projection'],FEATURES_DIR/'training_pca_clusters.csv')
plt.figure(figsize=(8,6)); plt.plot(result['scores'].k,result['scores'].inertia,marker='o'); plt.xlabel('k'); plt.ylabel('K-Means inertia'); plt.title('Elbow method'); plt.savefig(new_artifact_path(FIGURES_DIR/'08_elbow.png'),dpi=160,bbox_inches='tight'); plt.show()
plt.figure(figsize=(8,6)); plt.plot(result['scores'].k,result['scores'].silhouette,marker='o'); plt.xlabel('k'); plt.ylabel('Silhouette'); plt.title('Silhouette by k'); plt.savefig(new_artifact_path(FIGURES_DIR/'08_silhouette.png'),dpi=160,bbox_inches='tight'); plt.show()
plt.figure(figsize=(9,7)); plt.scatter(result['projection'].PC1,result['projection'].PC2,c=result['projection'].cluster,cmap='tab10',s=12); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Training cycles: unsupervised K-Means clusters'); plt.savefig(new_artifact_path(FIGURES_DIR/'08_pca_clusters.png'),dpi=160,bbox_inches='tight'); plt.show()
plt.figure(figsize=(9,7)); plt.scatter(result['projection'].PC1,result['projection'].PC2,c=pd.factorize(result['projection'].diagnosis)[0],cmap='tab20',s=12); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Training cycles: diagnoses shown after unsupervised fit'); plt.savefig(new_artifact_path(FIGURES_DIR/'08_pca_diagnosis.png'),dpi=160,bbox_inches='tight'); plt.show()
plt.figure(figsize=(8,5)); plt.plot(result['pca'].explained_variance_ratio_.cumsum()); plt.axhline(.95,color='red',linestyle='--'); plt.xlabel('Number of components'); plt.ylabel('Cumulative explained variance'); plt.savefig(new_artifact_path(FIGURES_DIR/'08_pca_cumulative_variance.png'),dpi=160,bbox_inches='tight'); plt.show()
print('ARI values are descriptive only and computed after clustering; they did not influence cluster fitting.')